In [ ]:
# ============================================================
# 셀 1 - 환경 설정 + DB 헬퍼 정의
# ============================================================
import os
import time
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from datetime import date
from dateutil.relativedelta import relativedelta
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = unquote(os.environ["MY_SERVICE_KEY"])

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저 (예외 시에도 안전)."""
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"SERVICE_KEY 로드됨 (길이 {len(SERVICE_KEY)})")
print(f"DB: {DB_PATH} (헬퍼 준비됨)")

In [ ]:
# ============================================================
# 셀 2 - Pydantic 모델 (공공데이터 표준 응답 + RailwayLineItem)
# ============================================================
# data_ingestion.ipynb / bus_route.ipynb의 PublicDataResponse와 동일 구조.
# 노트북 간 import가 번거로워 재정의. (TODO: 공통화 시 models.py로)

from typing import Generic, TypeVar, Optional
from pydantic import BaseModel, field_validator

T = TypeVar("T", bound=BaseModel)


class Header(BaseModel):
    resultCode: str
    resultMsg: str


class Items(BaseModel, Generic[T]):
    item: list[T] = []

    @field_validator("item", mode="before")
    @classmethod
    def _coerce_to_list(cls, v):
        if v is None or v == "":
            return []
        if isinstance(v, dict):
            return [v]
        return v


class Body(BaseModel, Generic[T]):
    items: Items[T] = Items[T]()
    dataType: str | None = None
    pageNo: int
    numOfRows: int
    totalCount: int


class Response_(BaseModel, Generic[T]):
    header: Header
    body: Body[T]


class PublicDataResponse(BaseModel, Generic[T]):
    Response: Response_[T]


class RailwayLineItem(BaseModel):
    """전철 호선 한 건.

    PK: (sarea_nm, rte_id)
      - 같은 rte_id(예: '001')가 다른 sarea(수도권/부산/대구)에서도 쓰일 수 있음
      - rte_nm은 노선명(예: '1호선')
      - dptre/arvl는 기점/종점명만. 여러 종점이 있을 경우 콤마 구분 (예: '인천,신창')
    """
    # 복합키 + 필수 필드
    sarea_nm: str                                # 서비스 지역명 (예: '수도권')
    rte_id: str                                  # 노선 ID (sarea 내 유일)
    opr_ymd: str                                 # 운행일자

    # Optional — API가 null 줄 수 있는 필드들
    rte_nm: Optional[str] = None                 # 노선명 (예: '1호선')
    dptre_sttn_nm: Optional[str] = None          # 기점명
    arvl_sttn_nm: Optional[str] = None           # 종점명 (복수일 땐 콤마 구분)


print("모델 로드 완료")

In [ ]:
# ============================================================
# 셀 3 - fetch 함수 (NO_DATA_FOUND 에러 처리 포함)
# ============================================================
# bus_route.ipynb의 fetch_all_pages와 동일.
# 데이터 없을 때 오는 {"Error": {"code": "50", "message": "NO_DATA_FOUND"}} 대응.

URBAN_RAILWAY_URL = "https://apis.data.go.kr/1613000/UrbanRailwayLine/getUrbanRailwayLine"


def fetch_all_pages(
    url: str,
    item_model: type[T],
    extra_params: dict | None = None,
    num_rows: int = 1000,
) -> list[T]:
    """공공데이터 표준 응답 API 전체 페이지 수집. NO_DATA_FOUND는 빈 결과로 취급."""
    params_base = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_rows,
        "dataType": "JSON",
        **(extra_params or {}),
    }
    all_items: list[T] = []
    page = 1
    while True:
        res = requests.get(url, params={**params_base, "pageNo": page}, timeout=30)
        res.raise_for_status()
        payload = res.json()

        # 에러 응답 선처리
        if "Error" in payload:
            err = payload["Error"]
            code = err.get("code")
            msg = err.get("message", "")
            if msg == "NO_DATA_FOUND" or code == "50":
                return all_items
            raise RuntimeError(f"API error: {code} {msg}")

        # Pydantic 검증
        parsed = PublicDataResponse[item_model].model_validate(payload)
        header = parsed.Response.header
        if header.resultCode not in ("00", "200"):
            raise RuntimeError(f"API error: {header.resultCode} {header.resultMsg}")

        body = parsed.Response.body
        items = body.items.item
        all_items.extend(items)
        print(f"  page {page}: +{len(items)}건 (누적 {len(all_items)}/{body.totalCount})")

        if len(all_items) >= body.totalCount or not items:
            break
        page += 1
        time.sleep(0.15)

    return all_items


print("fetch 함수 준비 완료")

In [ ]:
# ============================================================
# 셀 4 - 전국 전철 호선 수집
# ============================================================
# opr_ymd는 bus_route와 동일한 '오늘 - 1개월' 기준.
# 시도 루프 없이 한 번에 가져옴.
opr_ymd = (date.today() - relativedelta(months=1)).strftime("%Y%m%d")
print(f"운행일자 opr_ymd = {opr_ymd}")

lines = fetch_all_pages(
    URBAN_RAILWAY_URL,
    RailwayLineItem,
    extra_params={"opr_ymd": opr_ymd},
)

railway_df = pd.DataFrame([l.model_dump() for l in lines])
print(f"\n총 {len(railway_df)}건 수집")
print(railway_df.head(10))

In [ ]:
# ============================================================
# 셀 5 - DuckDB 적재 (railway_line 테이블, 복합 PK)
# ============================================================
# 중복 방어: API가 같은 (sarea_nm, rte_id) 쌍을 여러 번 반환할 가능성
before = len(railway_df)
railway_df_dedup = railway_df.drop_duplicates(subset=["sarea_nm", "rte_id"])
after = len(railway_df_dedup)
if before != after:
    print(f"⚠️  중복 (sarea_nm, rte_id) {before - after}건 제거")

with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS railway_line (
        sarea_nm       TEXT,
        rte_id         TEXT,
        opr_ymd        TEXT,
        rte_nm         TEXT,
        dptre_sttn_nm  TEXT,
        arvl_sttn_nm   TEXT,
        PRIMARY KEY (sarea_nm, rte_id)
    )
    """)

    con.execute("DELETE FROM railway_line")
    con.register("railway_view", railway_df_dedup)
    con.execute("""
    INSERT INTO railway_line
    SELECT sarea_nm, rte_id, opr_ymd, rte_nm, dptre_sttn_nm, arvl_sttn_nm
    FROM railway_view
    """)
    con.unregister("railway_view")

    cnt = con.execute("SELECT COUNT(*) FROM railway_line").fetchone()[0]

print(f"적재 완료: {cnt}건")

In [ ]:
# ============================================================
# 셀 6 - 검증
# ============================================================
with db_open(read_only=True) as con:
    print("=== 서비스 지역별 노선 수 ===")
    print(con.execute("""
        SELECT sarea_nm, COUNT(*) AS line_cnt
        FROM railway_line
        GROUP BY sarea_nm
        ORDER BY line_cnt DESC
    """).df())

    print("\n=== 수도권 노선 ===")
    print(con.execute("""
        SELECT rte_id, rte_nm, dptre_sttn_nm, arvl_sttn_nm
        FROM railway_line
        WHERE sarea_nm = '수도권'
        ORDER BY rte_id
    """).df())